# Appendix F: Special Data Structures in TensorFlow

In [1]:
import numpy as np
import tensorflow as tf

### Strings

In [2]:
tf.constant(b"Hello world!")

<tf.Tensor: shape=(), dtype=string, numpy=b'Hello world!'>

In [3]:
type(tf.constant(b"Hello world!"))

tensorflow.python.framework.ops.EagerTensor

In [4]:
tf.constant("Hello world!")

<tf.Tensor: shape=(), dtype=string, numpy=b'Hello world!'>

In [5]:
# tf.constant(b"café")    # SyntaxError: bytes can only contain ASCII literal characters.
tf.constant("café")

<tf.Tensor: shape=(), dtype=string, numpy=b'caf\xc3\xa9'>

In [6]:
tf.constant([ord(c) for c in "café"])

<tf.Tensor: shape=(4,), dtype=int32, numpy=array([ 99,  97, 102, 233], dtype=int32)>

In [7]:
str_utf8 = tf.constant("Hello café")
str_utf8

<tf.Tensor: shape=(), dtype=string, numpy=b'Hello caf\xc3\xa9'>

In [8]:
str_unicode = tf.strings.unicode_decode(str_utf8, "UTF-8")
str_unicode

<tf.Tensor: shape=(10,), dtype=int32, numpy=array([ 72, 101, 108, 108, 111,  32,  99,  97, 102, 233], dtype=int32)>

In [9]:
tf.strings.length(str_utf8)

<tf.Tensor: shape=(), dtype=int32, numpy=11>

In [10]:
caffee = tf.constant(["Café", "Coffee", "caffè", "咖啡"])
caffee

<tf.Tensor: shape=(4,), dtype=string, numpy=
array([b'Caf\xc3\xa9', b'Coffee', b'caff\xc3\xa8',
       b'\xe5\x92\x96\xe5\x95\xa1'], dtype=object)>

In [11]:
tf.strings.length(caffee)

<tf.Tensor: shape=(4,), dtype=int32, numpy=array([5, 6, 6, 6], dtype=int32)>

In [12]:
tf.strings.length(caffee, "UTF8_CHAR")

<tf.Tensor: shape=(4,), dtype=int32, numpy=array([4, 6, 5, 2], dtype=int32)>

### Ragged Tensors

In [13]:
t_ragged = tf.strings.unicode_decode(caffee, "UTF8")
t_ragged

<tf.RaggedTensor [[67, 97, 102, 233], [67, 111, 102, 102, 101, 101], [99, 97, 102, 102, 232], [21654, 21857]]>

In [14]:
t1_ragged = tf.ragged.constant([[1, 2, 3], [4, 5]])
t1_ragged

<tf.RaggedTensor [[1, 2, 3], [4, 5]]>

In [15]:
tf.concat([t_ragged, t1_ragged], axis=0)

<tf.RaggedTensor [[67, 97, 102, 233], [67, 111, 102, 102, 101, 101], [99, 97, 102, 102, 232], [21654, 21857], [1, 2, 3], [4, 5]]>

In [16]:
t_ragged.to_tensor()

<tf.Tensor: shape=(4, 6), dtype=int32, numpy=
array([[   67,    97,   102,   233,     0,     0],
       [   67,   111,   102,   102,   101,   101],
       [   99,    97,   102,   102,   232,     0],
       [21654, 21857,     0,     0,     0,     0]], dtype=int32)>

### Sparse Tensors

In [17]:
a = np.zeros((2, 3))
a[[0, 1], [1, 2]] = [1, 2]
a

array([[0., 1., 0.],
       [0., 0., 2.]])

In [18]:
t = tf.constant(a)
t

<tf.Tensor: shape=(2, 3), dtype=float64, numpy=
array([[0., 1., 0.],
       [0., 0., 2.]])>

In [19]:
# t_sparse = tf.sparse.SparseTensor([[0, 1], [1, 0], [2, 3]], values=[1, 2, 3], dense_shape=[3, 4])
t_sparse = tf.sparse.SparseTensor(
    [[0, 1], [2, 3], [1, 0]], values=[1, 2, 3], dense_shape=[3, 4]
)
t_sparse

In [20]:
t_sparse = tf.sparse.reorder(t_sparse)
tf.sparse.to_dense(t_sparse)

<tf.Tensor: shape=(3, 4), dtype=int32, numpy=
array([[0, 1, 0, 0],
       [3, 0, 0, 0],
       [0, 0, 0, 2]], dtype=int32)>

### Tensor Arrays

In [21]:
tarr = tf.TensorArray(dtype=tf.float32, size=3)
tarr = tarr.write(0, tf.constant([1, 2.0]))
tarr = tarr.write(1, tf.constant([3, 10.0]))
tarr = tarr.write(2, tf.constant([5, 7.0]))
tarr

In [22]:
tarr.stack()

<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
array([[ 1.,  2.],
       [ 3., 10.],
       [ 5.,  7.]], dtype=float32)>

### Sets

In [23]:
s1 = tf.constant([[1, 2, 3]])
s2 = tf.constant([[2, 3, 4, 5]])
tf.sparse.to_dense(tf.sets.difference(s2, s1))

<tf.Tensor: shape=(1, 2), dtype=int32, numpy=array([[4, 5]], dtype=int32)>

In [24]:
s = tf.sparse.SparseTensor([[0, 3]], [3], [1, 4])
tf.sparse.to_dense(s)

<tf.Tensor: shape=(1, 4), dtype=int32, numpy=array([[0, 0, 0, 3]], dtype=int32)>

In [25]:
tf.sets.size(s)

<tf.Tensor: shape=(1,), dtype=int32, numpy=array([1], dtype=int32)>

### Queues

In [26]:
q = tf.queue.PriorityQueue(capacity=3, types=[tf.string, tf.float32], shapes=[(), ()])
q

In [27]:
q.enqueue_many([[10, 3, 100], [b"sunny", b"windy", b"cloudy"], [1.0, 2.0, 3.0]])
q

In [28]:
q.dequeue()

[<tf.Tensor: shape=(), dtype=int64, numpy=3>,
 <tf.Tensor: shape=(), dtype=string, numpy=b'windy'>,
 <tf.Tensor: shape=(), dtype=float32, numpy=2.0>]

In [29]:
# q.enqueue_many([[50, 51], [b"rainy", b"shower"], [4, 5]])

In [ ]:
# q.dequeue_many(4)